# SecurityAI — QLoRA Fine-Tuning Notebook

Fine-tunes **Phi-3 Mini (3.8B)** on the SecurityAI QA dataset using Unsloth + QLoRA.
Exports a GGUF file ready to drop into Ollama.

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free Colab tier is enough)
2. Upload `train.jsonl` and `test.jsonl` from `backend/data/dataset/` to this Colab session
3. Run all cells in order

### What this notebook does
| Step | What happens |
|---|---|
| 1 | Install Unsloth + training deps |
| 2 | Load Phi-3 Mini in 4-bit (fits in 8 GB VRAM) |
| 3 | Attach QLoRA adapters (16-rank) |
| 4 | Train on `train.jsonl` for 1 epoch |
| 5 | Evaluate perplexity on `test.jsonl` |
| 6 | Export merged model as `securityai.Q4_K_M.gguf` |
| 7 | Download GGUF + instructions to load into Ollama |

## Step 1 — Install dependencies

Unsloth patches the Hugging Face stack for 2× faster training and 60% less memory.  
This cell takes ~3 minutes on a fresh Colab runtime.

In [ ]:
%%capture
import sys

# Unsloth — installs the right CUDA-matched torch wheel automatically
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Configuration

Edit these values if you want to experiment with different models or hyperparameters.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL      = "unsloth/Phi-3-mini-4k-instruct"   # 3.8B, fits in T4 (16 GB)
# Alternative: "unsloth/Llama-3.2-3B-Instruct"       # also fits in T4

MAX_SEQ_LENGTH  = 2048    # Phi-3 Mini native context; bump to 4096 if on A100
LOAD_IN_4BIT    = True    # QLoRA — quantize base weights to 4-bit

# ── LoRA adapters ─────────────────────────────────────────────────────────────
LORA_RANK       = 16      # Higher = more capacity, more VRAM. 16 is a good default.
LORA_ALPHA      = 16      # Usually equal to rank

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS      = 1       # 1–2 epochs is enough for fine-tuning on 3k+ examples
BATCH_SIZE      = 2       # Per-device batch size
GRAD_ACCUM      = 4       # Effective batch = BATCH_SIZE × GRAD_ACCUM = 8
LEARNING_RATE   = 2e-4
OUTPUT_DIR      = "securityai-checkpoints"

# ── Dataset ───────────────────────────────────────────────────────────────────
TRAIN_FILE      = "train.jsonl"   # upload these to Colab before running
TEST_FILE       = "test.jsonl"

# ── Export ────────────────────────────────────────────────────────────────────
EXPORT_NAME     = "securityai"
QUANT_METHOD    = "q4_k_m"        # Good balance of quality vs size (~2.4 GB)
# Alternatives: "q5_k_m" (better quality, ~3 GB), "q8_0" (best quality, ~4 GB)

## Step 3 — Load base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect: bf16 on Ampere+, fp16 otherwise
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Step 4 — Attach QLoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    # Attention + MLP projections — standard set for instruction tuning
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,        # 0 is optimal for QLoRA per Unsloth benchmarks
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimised checkpointing
    random_state=42,
    use_rslora=False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of total)")

## Step 5 — Load and format dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": TRAIN_FILE, "test": TEST_FILE})
print(f"Train examples : {len(raw['train'])}")
print(f"Test examples  : {len(raw['test'])}")
print("\nSample training example:")
print(raw["train"][0])

In [ ]:
# Apply the model's native chat template to each example.
# Our dataset is already in ChatML messages format so this maps directly.

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


dataset = raw.map(format_example, remove_columns=["messages"])

# Quick sanity check — print one formatted training example
print(dataset["train"][0]["text"][:600], "...")

## Step 6 — Train

Expected time on a free T4:
- ~3,200 training examples, 1 epoch → **~40–60 minutes**

Watch the `loss` column — it should decrease from ~2.0 to ~1.0–1.3 over the run.  
If loss plateaus above 1.5 after 100 steps, try `LEARNING_RATE = 1e-4`.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,    # packing can save time but may hurt instruction-following quality
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.03,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        output_dir=OUTPUT_DIR,
        report_to="none",          # set to "wandb" if you want experiment tracking
        seed=42,
    ),
)

# Show memory usage before training
gpu_stats = torch.cuda.get_device_properties(0)
reserved  = torch.cuda.memory_reserved() / 1e9
print(f"GPU: {gpu_stats.name}  |  Reserved: {reserved:.1f} GB / {gpu_stats.total_memory/1e9:.1f} GB")

In [ ]:
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Runtime   : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Samples/s : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 7 — Evaluate on test set

Measures perplexity on held-out examples.  
A good fine-tuned security model should achieve perplexity **< 4.0** on this domain.

In [ ]:
import math
from transformers import Trainer, TrainingArguments as EvalArgs

FastLanguageModel.for_inference(model)   # switch to faster inference mode

eval_trainer = Trainer(
    model=model,
    args=EvalArgs(
        output_dir="eval_tmp",
        per_device_eval_batch_size=2,
        report_to="none",
    ),
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
)

results = eval_trainer.evaluate()
perplexity = math.exp(results["eval_loss"])
print(f"Eval loss  : {results['eval_loss']:.4f}")
print(f"Perplexity : {perplexity:.2f}")

## Step 8 — Manual inference test

Sanity-check the model on a few security questions before exporting.

In [ ]:
from transformers import TextStreamer

SYSTEM = (
    "You are SecurityAI, an expert cybersecurity assistant specializing in "
    "secure application development, penetration testing, and reverse engineering. "
    "You provide accurate, practical guidance grounded in authoritative security literature."
)

TEST_QUESTIONS = [
    "What is a heap spray attack and how does it help exploit browser vulnerabilities?",
    "Walk me through using Metasploit to perform a basic exploitation workflow.",
    "How do stack canaries protect against buffer overflow attacks?",
]

streamer = TextStreamer(tokenizer, skip_prompt=True)

for question in TEST_QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    messages = [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    _ = model.generate(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        use_cache=True,
    )

## Step 9 — Export to GGUF

Merges the LoRA adapters back into the base weights, then quantizes to GGUF.  
The resulting file is what Ollama loads — no other conversion needed.

This cell takes **5–10 minutes**.

In [ ]:
# Merge adapters + quantize in one step
model.save_pretrained_gguf(
    EXPORT_NAME,
    tokenizer,
    quantization_method=QUANT_METHOD,
)

import os, glob
gguf_files = glob.glob(f"{EXPORT_NAME}*.gguf")
for f in gguf_files:
    size_gb = os.path.getsize(f) / 1e9
    print(f"Exported: {f}  ({size_gb:.2f} GB)")

## Step 10 — Write the Ollama Modelfile

In [ ]:
import glob

gguf_file = glob.glob(f"{EXPORT_NAME}*.gguf")[0]

modelfile_content = f"""FROM ./{gguf_file}

SYSTEM \"\"\"
You are SecurityAI, an expert cybersecurity assistant specializing in secure
application development, penetration testing, and reverse engineering. You
provide accurate, practical guidance grounded in authoritative security
literature including Gray Hat Hacking and Reversing: Secrets of Reverse
Engineering.
\"\"\"

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|end|>"
PARAMETER stop "<|user|>"
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile written:")
print(modelfile_content)

## Step 11 — Download files from Colab

Download the GGUF and Modelfile to your local machine.

In [ ]:
from google.colab import files
import glob

# Download Modelfile first (tiny)
files.download("Modelfile")

# Download GGUF (2–4 GB — may take a few minutes)
for gguf in glob.glob(f"{EXPORT_NAME}*.gguf"):
    print(f"Downloading {gguf} ...")
    files.download(gguf)

## Step 12 — Load into Ollama (run locally, not in Colab)

After downloading, run these commands on your local machine:

```bash
# 1. Move files into the backend models directory
mv securityai-unsloth.Q4_K_M.gguf SecurityAI/backend/models/
mv Modelfile SecurityAI/backend/models/

# 2. Register the model with Ollama
cd SecurityAI/backend/models
ollama create securityai -f Modelfile

# 3. Test it
ollama run securityai "What is a buffer overflow?"

# 4. Update backend/.env to use the new model
#    OLLAMA_MODEL=securityai

# 5. Restart the stack
docker compose up --build
```

### Docker — update the model in docker-compose.yml

```yaml
# In docker-compose.yml, change:
OLLAMA_MODEL: ${OLLAMA_MODEL:-securityai}

# And in the ollama-init service entrypoint, the model pull will be skipped
# since securityai is a locally-created model. You can comment out
# the ollama-init service once the GGUF is loaded into the ollama volume.
```

### What changes in the backend

Nothing else changes — the FastAPI backend still talks to Ollama on port 11434  
using the same streaming API. Only `OLLAMA_MODEL` needs updating.

The RAG pipeline (ChromaDB retrieval) continues to run alongside the  
fine-tuned model for the **outcomes collection** (continual learning).  
Textbook RAG can be disabled or kept as a secondary source — the model  
now has the textbook knowledge baked into its weights.

## Reference — Training hyperparameter guide

| Hyperparameter | Default | When to change |
|---|---|---|
| `LORA_RANK` | 16 | Increase to 32/64 for more model capacity (needs more VRAM) |
| `NUM_EPOCHS` | 1 | Increase to 2–3 if loss hasn't plateaued; risk of overfitting |
| `LEARNING_RATE` | 2e-4 | Decrease to 1e-4 if training is unstable |
| `BATCH_SIZE` | 2 | Decrease to 1 if OOM errors occur |
| `GRAD_ACCUM` | 4 | Increase to compensate for smaller batch sizes |
| `QUANT_METHOD` | q4_k_m | `q5_k_m` for better quality (+0.6 GB), `q8_0` for best (+1.5 GB) |

### Signs of a good training run
- Loss drops from ~2.0 → ~1.0–1.3 over 1 epoch
- Test perplexity < 4.0
- Inference answers are technically accurate and use correct security terminology

### Signs to re-run with adjustments
- Loss stuck above 1.8 → try lower learning rate (`1e-4`)
- Loss drops to 0.3–0.5 → likely overfitting, reduce epochs or add dropout
- Answers are generic or hallucinated → dataset quality issue, check `raw_qa_pairs.jsonl`